In [4]:
# Imports
import glob
import os
import re
import h5py
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"text.usetex": True})
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["mathtext.fontset"] = "custom"
plt.rcParams["mathtext.rm"] = "Times New Roman"
plt.rcParams["mathtext.fallback"] = "stix"
plt.style.use("seaborn-v0_8-deep")
COLORS  = plt.rcParams["axes.prop_cycle"].by_key()["color"]
MARKERS = ["o", "s", "^", "D"]

In [ ]:
"Change this to the directory where data is stored"

results_dir = r"C:\Users\harri290\Downloads"

In [2]:
# Parameters
# One batch folder per frequency we want to compare
batches = ["L32_OBC_m400_t11.0_g0.2_om5.0", "L32_OBC_m400_t11.0_g0.2_om10.0"]

# Matching ground-state energy tables (one per batch/frequency) for the excess-energy plot
gs_tables = ["L32_OBC_gs_table_t11.0_g0.2_om5.0", "L32_OBC_gs_table_t11.0_g0.2_om10.0"]

i_ref = 0 # reference site for correlation plots (0-based)

# Which file to load for all plots
g_plot    = 0.2
site_plot = 16

In [5]:
# File discovery
def discover_files(base_dir):
    index = {}
    for fpath in glob.glob(os.path.join(base_dir, "**", "output.h5"), recursive=True):
        folder = os.path.basename(os.path.dirname(fpath))
        gm  = re.search(r"g(-?[\d.]+)", folder)
        pm  = re.search(r"p(even|odd)", folder)
        sm  = re.search(r"site(\d+)", folder)
        if gm and pm and sm:
            g       = float(gm.group(1))
            parity  = 1 if pm.group(1) == "even" else -1
            ac_site = int(sm.group(1))
            index[(g, parity, ac_site)] = fpath
    return index

# One index per batch (frequency)
indices = {}
for batch in batches:
    base_dir = os.path.join(results_dir, batch)
    indices[batch] = discover_files(base_dir)
    print(f"{batch}: found {len(indices[batch])} files")
    for key in sorted(indices[batch]):
        print(f"  g={key[0]}, parity={'even' if key[1]==1 else 'odd'}, ac_site={key[2]}")

L32_OBC_m400_t11.0_g0.2_om5.0: found 0 files
L32_OBC_m400_t11.0_g0.2_om10.0: found 0 files


In [6]:
# Load HDF5 files
def load_file(fpath):
    with h5py.File(fpath, "r") as f:
        d = {}
        # Metadata
        d["L"]                = int(f["L"][()])
        d["bc"]               = str(f["bc"][()])
        d["parity"]           = int(f["parity"][()])
        d["t1"]               = float(f["t1"][()])
        d["g"]                = float(f["g"][()])
        d["omega"]            = float(f["omega"][()])
        d["periods"]          = int(f["periods"][()])
        d["steps_per_period"] = int(f["steps_per_period"][()])
        d["ac_site"]          = int(f["ac_site"][()])
        d["E0_dmrg"]          = float(f["E0_dmrg"][()])
        # Time axes
        d["times"]      = f["times"][:]
        d["meas_times"] = f["meas_times"][:]
        # Scalar time series
        d["loschmidt"] = f["loschmidt"][:]
        d["energy_t"]  = f["energy_t"][:]
        d["energy_0"]  = f["energy_0"][:]
        d["maxdims"]   = f["maxdims"][:]
        d["entropy_t"] = f["entropy_t"][:] if "entropy_t" in f else None
        # Autocorrelations
        d["autoc_x"] = f["re_autoc_x"][:] + 1j*f["im_autoc_x"][:]
        d["autoc_y"] = f["re_autoc_y"][:] + 1j*f["im_autoc_y"][:]
        d["autoc_z"] = f["re_autoc_z"][:] + 1j*f["im_autoc_z"][:]
        # Correlations: Julia writes (L, nmeas) and (L, L, nmeas)
        # so we transpose back to (nmeas, L) and (nmeas, L, L)
        d["X_t"]  = f["X_t"][:].T
        d["Y_t"]  = f["Y_t"][:].T
        d["Z_t"]  = f["Z_t"][:].T
        d["XX_t"] = f["XX_t"][:].transpose(2, 0, 1)
        d["YY_t"] = f["YY_t"][:].transpose(2, 0, 1)
        d["ZZ_t"] = f["ZZ_t"][:].transpose(2, 0, 1)
    # Connected correlations
    X, Y, Z = d["X_t"], d["Y_t"], d["Z_t"]
    d["XXc_t"] = d["XX_t"] - X[:, :, None] * X[:, None, :]
    d["YYc_t"] = d["YY_t"] - Y[:, :, None] * Y[:, None, :]
    d["ZZc_t"] = d["ZZ_t"] - Z[:, :, None] * Z[:, None, :]
    return d

# Load both parity sectors for each batch (frequency)
# runs[batch] = {"omega", "T", "t_over_T", "parities": [(d, label, linestyle), ...]}
runs = {}
for batch in batches:
    idx    = indices[batch]
    d_even = load_file(idx[(g_plot, 1,  site_plot)])
    d_odd  = load_file(idx[(g_plot, -1, site_plot)])
    T      = 2*np.pi / d_even["omega"]
    runs[batch] = {
        "omega":    d_even["omega"],
        "T":        T,
        "t_over_T": d_even["times"] / T,
        "parities": [(d_even, "even", "-"), (d_odd, "odd", "--")],
    }
    print(f"{batch}: omega={d_even['omega']}, T={T:.4f}, periods={d_even['periods']}")

KeyError: (0.2, 1, 16)

In [7]:
# Excess energy vs t/T (frequency comparison)

# Build energy lookup table: (t1_val, parity) -> E0
def build_E0_table(table_dir):
    table = {}
    for fpath in glob.glob(os.path.join(table_dir, "**", "output.h5"), recursive=True):
        folder = os.path.basename(os.path.dirname(fpath))
        tm = re.search(r"t1(-?[\d.]+)", folder)
        pm = re.search(r"p(even|odd)", folder)
        if not tm or not pm:
            continue
        t1_val = float(tm.group(1))
        parity = 1 if pm.group(1) == "even" else -1
        with h5py.File(fpath, "r") as f:
            E0 = float(f["energies"][0])
        table[(t1_val, parity)] = E0
    return table

# One ground-state energy table per batch (frequency)
E0_tables = {batch: build_E0_table(os.path.join(results_dir, gs_table))
             for batch, gs_table in zip(batches, gs_tables)}

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)

for b_i, batch in enumerate(batches):
    run      = runs[batch]
    E0_table = E0_tables[batch]
    t_over_T = run["t_over_T"]
    for d_, p_label, ls in run["parities"]:
        # Instantaneous ground-state energy follows the driven hopping t1*cos(omega t)
        t1_eff  = np.round(d_["t1"] * np.cos(d_["omega"] * d_["times"]), 10)
        missing = [v for v in t1_eff if (v, d_["parity"]) not in E0_table]
        if missing:
            print(f"No gs table match for {batch} {p_label} parity "
                  f"({len(set(missing))} t1 values missing) — "
                  f"run gs_table_setup.py with matching parameters")
            continue
        E0_t     = np.array([E0_table[(v, d_["parity"])] for v in t1_eff])
        excess_E = d_["energy_t"] - E0_t
        ax.plot(t_over_T, excess_E, linestyle=ls, color=COLORS[b_i],
                linewidth=1.5, marker='.', markersize=3,
                label=rf"$\omega={d_['omega']}$ ({p_label})")

ax.set_xlabel(r"$t/T$", fontsize=12)
ax.set_ylabel(r"$\langle \psi(t) | H(t) | \psi(t) \rangle - E_0(t)$", fontsize=12)
ax.set_title(
    rf"Excess Energy, $g/t={g_plot}$, $L={runs[batches[0]]['parities'][0][0]['L']}$",
    fontsize=13
)
ax.set_xlim(left=0)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale("log")
plt.tight_layout()
plt.show()

KeyError: 'L32_OBC_m400_t11.0_g0.2_om5.0'

Error in callback <function _draw_all_if_interactive at 0x000001C8A420DC60> (for post_execute), with arguments args (),kwargs {}:


RuntimeError: Failed to process string with tex because latex could not be found

RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 1200x600 with 1 Axes>

In [ ]:
# Autocorrelation vs t/T (frequency comparison)
# Autocorrelation time series are already loaded per file as d["autoc_x/y/z"]
# (complex), so no lookup table is needed here.

ops     = ["X", "Y", "Z"]
ac_keys = ["autoc_x", "autoc_y", "autoc_z"]

for p_idx, p_label in enumerate(["even", "odd"]):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)
    site = runs[batches[0]]["parities"][p_idx][0]["ac_site"]
    # Overlay each frequency: color = frequency, solid = Re, dashed = Im
    for b_i, batch in enumerate(batches):
        run      = runs[batch]
        d_       = run["parities"][p_idx][0]
        t_over_T = run["t_over_T"]
        for ax, op, key in zip(axes, ops, ac_keys):
            autoc = d_[key]
            ax.plot(t_over_T, np.real(autoc), "-",  color=COLORS[b_i], linewidth=1.5,
                    marker='.', markersize=3, label=rf"Re, $\omega={d_['omega']}$")
            ax.plot(t_over_T, np.imag(autoc), "--", color=COLORS[b_i], linewidth=1.5,
                    marker='.', markersize=3, label=rf"Im, $\omega={d_['omega']}$")
    for ax, op in zip(axes, ops):
        ax.set_xlabel(r"$t/T$", fontsize=12)
        ax.set_ylabel(rf"$\langle {op}_{{{site}}}(t){op}_{{{site}}}(0)\rangle$", fontsize=12)
        ax.set_title(rf"{op} Autocorrelation", fontsize=13)
        ax.set_xlim(left=0)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    plt.suptitle(
        rf"site = {site}, $g/t={g_plot}$, $L={runs[batches[0]]['parities'][0][0]['L']}$ ({p_label} parity)",
        fontsize=13, y=1.02
    )
    plt.tight_layout()
    plt.show()